# 06 - Dashboard exploratorio de ingresos

Dashboard para explorar relaciones entre ingresos y variables demográficas, laborales, de hogar, vivienda y geografía. Funciona dentro de Jupyter con `ipywidgets`, `pandas`, `numpy`, `seaborn` y `matplotlib`.

Nota: municipio se usa de forma exploratoria; no implica representatividad municipal.

## Dependencias del kernel

Ejecuta esta celda antes del dashboard. Verifica e instala las dependencias faltantes en el kernel activo de Jupyter; esto evita instalar paquetes en otro Python distinto al que esta corriendo el notebook.


In [ ]:
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = {
    "ipywidgets": "ipywidgets",
    "seaborn": "seaborn",
    "matplotlib": "matplotlib",
}

missing = []
for module_name, package_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        missing.append(package_name)

if missing:
    print(f"Instalando dependencias faltantes en este kernel: {', '.join(missing)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    importlib.invalidate_caches()

still_missing = []
for module_name, package_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        still_missing.append(package_name)

if still_missing:
    raise RuntimeError(
        "No se pudieron importar estas dependencias en el kernel activo: "
        + ", ".join(still_missing)
        + ". Instalalas manualmente con: "
        + f"{sys.executable} -m pip install "
        + " ".join(still_missing)
    )

print("Dependencias listas en:", sys.executable)


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

try:
    import ipywidgets as widgets
    import seaborn as sns
    import matplotlib
    from IPython import get_ipython

    _ipython = get_ipython()
    if _ipython is not None:
        try:
            _ipython.run_line_magic("matplotlib", "inline")
        except Exception as backend_error:
            print(f"No pude activar matplotlib inline; usare display(fig). Detalle: {backend_error}")
    else:
        matplotlib.use("Agg")

    import matplotlib.pyplot as plt
    from IPython.display import display, clear_output
except ImportError as exc:
    raise RuntimeError(
        "Faltan dependencias en el kernel activo. Ejecuta primero la celda "
        "'Dependencias del kernel' o instala manualmente con: "
        f"{sys.executable} -m pip install matplotlib seaborn ipywidgets"
    ) from exc

DASHBOARD_READY = True


def render_figure(fig):
    display(fig)
    plt.close(fig)


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("No pude localizar la raiz del proyecto.")


ROOT = find_project_root()
REV4 = ROOT / "data" / "interim" / "revision_4"
PERSON_PATH = REV4 / "mart_persona_2018_2024.csv.gz"
HOUSE_PATH = REV4 / "mart_hogar_2018_2024.csv.gz"

print(f"Proyecto: {ROOT}")
print(f"Dashboard listo: {DASHBOARD_READY}")


## 1. Cargar marts

El dashboard usa los marts ya validados en el notebook 05. Para rendimiento, se cargan una vez y los filtros trabajan sobre memoria.

In [ ]:
mart_persona = pd.read_csv(PERSON_PATH, low_memory=False)
mart_hogar = pd.read_csv(HOUSE_PATH, low_memory=False)
diccionario = pd.read_csv(REV4 / "diccionario_marts.csv")

for df in [mart_persona, mart_hogar]:
    if "anio" in df.columns:
        df["anio"] = pd.to_numeric(df["anio"], errors="coerce").astype("Int64")

print(f"mart_persona: {mart_persona.shape}")
print(f"mart_hogar: {mart_hogar.shape}")

## 2. Configuración de variables

In [ ]:
TARGETS = {
    "Persona": [
        "ingreso_persona_total_registros_tri",
        "ingreso_persona_laboral_negocio_tri",
        "ingreso_persona_transferencias_tri",
        "ingreso_persona_rentas_propiedad_tri",
        "ing_cor_hogar_pc_oficial_tri",
        "ingtrab_hogar_pc_oficial_tri",
    ],
    "Hogar": [
        "ing_cor_hogar_oficial_tri",
        "ingtrab_hogar_oficial_tri",
        "ing_cor_pc_oficial_tri",
        "ingtrab_pc_oficial_tri",
        "ingreso_personas_total_registros_tri",
        "ingreso_personas_laboral_negocio_tri",
        "ingreso_personas_transferencias_tri",
    ],
}

VARIABLE_GROUPS = {
    "Demografía": ["edad", "sexo_desc", "edo_conyug_desc", "etnia_desc"],
    "Educación": ["nivel_desc", "nivelaprob_desc", "alfabetism_desc", "asis_esc_desc"],
    "Trabajo": ["n_trabajos", "horas_trabajos_total", "pago_principal_desc", "contrato_principal_desc", "tiene_suel_principal_desc"],
    "Hogar": ["tot_integ", "ocupados", "clase_hog_desc", "sexo_jefe_desc", "educa_jefe_desc", "conex_inte_desc_hogar"],
    "Vivienda": ["tipo_viv_desc_vivienda", "tenencia_desc_vivienda", "num_cuarto_vivienda", "drenaje_desc_vivienda", "disp_elect_desc_vivienda"],
    "Geografía": ["entidad", "municipio", "tam_loc_desc", "est_socio_desc"],
}


def available_targets(level):
    df = mart_persona if level == "Persona" else mart_hogar
    return [col for col in TARGETS[level] if col in df.columns]


def available_xvars(level):
    df = mart_persona if level == "Persona" else mart_hogar
    values = []
    for group, cols in VARIABLE_GROUPS.items():
        for col in cols:
            if col in df.columns:
                values.append((f"{group} | {col}", col))
    return values


def detect_var_type(series):
    numeric = pd.to_numeric(series, errors="coerce")
    numeric_ratio = numeric.notna().mean()
    nunique = series.nunique(dropna=True)
    if numeric_ratio > 0.9 and nunique > 20:
        return "numérica continua"
    if numeric_ratio > 0.9:
        return "numérica discreta"
    return "categórica"


def filter_data(df, year, entidad, municipio, tam_loc, est_socio, min_obs):
    out = df.copy()
    if year != "Todos":
        out = out[out["anio"].astype(str).eq(str(year))]
    for col, value in [("entidad", entidad), ("municipio", municipio), ("tam_loc_desc", tam_loc), ("est_socio_desc", est_socio)]:
        if value != "Todos" and col in out.columns:
            out = out[out[col].astype(str).eq(str(value))]
    return out

## 3. Funciones de análisis

In [ ]:
def income_distribution(df, y):
    data = pd.to_numeric(df[y], errors="coerce").dropna()
    stats = data.quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).rename("valor").reset_index().rename(columns={"index": "percentil"})
    stats.loc[len(stats)] = ["media", data.mean()]
    display(stats)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    sns.histplot(data.sample(min(len(data), 80_000), random_state=42), bins=50, kde=True, ax=axes[0])
    axes[0].set_title(f"Distribución de {y}")
    axes[0].set_xlabel(y)
    sns.boxplot(x=data.sample(min(len(data), 80_000), random_state=42), ax=axes[1])
    axes[1].set_title("Boxplot")
    plt.tight_layout()
    render_figure(fig)


def x_vs_income(df, x, y, max_categories=15):
    work = df[[x, y]].dropna().copy()
    work[y] = pd.to_numeric(work[y], errors="coerce")
    work = work.dropna(subset=[y])
    var_type = detect_var_type(work[x])
    print(f"Tipo detectado: {var_type}; n válido={len(work):,}")

    if var_type.startswith("numérica"):
        work[x] = pd.to_numeric(work[x], errors="coerce")
        work = work.dropna(subset=[x])
        pearson = work[[x, y]].corr(method="pearson").iloc[0, 1]
        spearman = work[[x, y]].corr(method="spearman").iloc[0, 1]
        print(f"Pearson={pearson:.3f}; Spearman={spearman:.3f}")
        sample = work.sample(min(len(work), 20_000), random_state=42)
        fig, ax = plt.subplots(figsize=(7, 4.5))
        sns.scatterplot(data=sample, x=x, y=y, s=10, alpha=0.25, ax=ax)
        binned = work.assign(_bin=pd.qcut(work[x], q=20, duplicates="drop")).groupby("_bin", observed=True).agg(x_med=(x, "median"), y_med=(y, "median")).reset_index()
        ax.plot(binned["x_med"], binned["y_med"], color="crimson", linewidth=2, label="mediana por bins")
        ax.legend()
        ax.set_title(f"{y} vs {x}")
        plt.tight_layout()
        render_figure(fig)
    else:
        counts = work[x].astype(str).value_counts().head(max_categories).index
        work = work[work[x].astype(str).isin(counts)]
        summary = work.groupby(x)[y].agg(n="size", media="mean", mediana="median", p25=lambda s: s.quantile(0.25), p75=lambda s: s.quantile(0.75)).reset_index().sort_values("mediana", ascending=False)
        display(summary)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
        sns.boxplot(data=work, x=y, y=x, order=summary[x], ax=axes[0])
        sns.barplot(data=summary, x="mediana", y=x, ax=axes[1])
        axes[0].set_title(f"Distribución de {y} por {x}")
        axes[1].set_title("Mediana por categoría")
        plt.tight_layout()
        render_figure(fig)


def time_evolution(df, y, segment):
    work = df[["anio", y, segment]].dropna().copy()
    work[y] = pd.to_numeric(work[y], errors="coerce")
    work = work.dropna(subset=[y])
    top = work[segment].astype(str).value_counts().head(8).index
    work = work[work[segment].astype(str).isin(top)]
    summary = work.groupby(["anio", segment])[y].agg(n="size", mediana="median").reset_index()
    display(summary)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.lineplot(data=summary, x="anio", y="mediana", hue=segment, marker="o", ax=ax)
    ax.set_title(f"Evolución de {y} por {segment}")
    plt.tight_layout()
    render_figure(fig)


def geography_view(df, y, geo, min_obs):
    work = df[[geo, y]].dropna().copy()
    work[y] = pd.to_numeric(work[y], errors="coerce")
    work = work.dropna(subset=[y])
    summary = work.groupby(geo)[y].agg(n="size", media="mean", mediana="median", p25=lambda s: s.quantile(0.25), p75=lambda s: s.quantile(0.75)).reset_index()
    summary = summary[summary["n"] >= min_obs].sort_values("mediana", ascending=False).head(25)
    display(summary)
    fig, ax = plt.subplots(figsize=(9, 6))
    sns.barplot(data=summary, x="mediana", y=geo, ax=ax)
    ax.set_title(f"{y} por {geo} (n mínimo={min_obs})")
    plt.tight_layout()
    render_figure(fig)

## 4. Dashboard

In [ ]:
if DASHBOARD_READY:
    level = widgets.ToggleButtons(options=["Persona", "Hogar"], description="Nivel")
    target = widgets.Dropdown(description="Ingreso")
    year = widgets.Dropdown(options=["Todos", "2018", "2020", "2022", "2024"], value="Todos", description="Año")
    entidad = widgets.Dropdown(description="Entidad")
    municipio = widgets.Dropdown(description="Municipio")
    tam_loc = widgets.Dropdown(description="Tam. loc.")
    est_socio = widgets.Dropdown(description="Estrato")
    min_obs = widgets.IntSlider(value=30, min=1, max=500, step=1, description="n mínimo")
    xvar = widgets.Dropdown(description="X")
    segment = widgets.Dropdown(description="Segmento")
    view = widgets.Dropdown(options=["Distribución", "X vs ingreso", "Evolución temporal", "Geografía"], description="Vista")
    max_categories = widgets.IntSlider(value=15, min=5, max=40, step=1, description="Categorías")
    output = widgets.Output()
    _INITIALIZING = False

    def current_df():
        return mart_persona if level.value == "Persona" else mart_hogar

    def refresh_options(*_):
        global _INITIALIZING
        _INITIALIZING = True
        df = current_df()
        targets = available_targets(level.value)
        x_options = available_xvars(level.value)
        target.options = targets
        target.value = targets[0] if targets else None
        xvar.options = x_options
        xvar.value = x_options[0][1] if x_options else None
        segment.options = [("Sin segmento", None)] + x_options
        segment.value = None
        for widget, col in [(entidad, "entidad"), (municipio, "municipio"), (tam_loc, "tam_loc_desc"), (est_socio, "est_socio_desc")]:
            if col in df.columns:
                vals = ["Todos"] + sorted(df[col].dropna().astype(str).unique().tolist())
                widget.options = vals
                widget.value = "Todos"
            else:
                widget.options = ["Todos"]
                widget.value = "Todos"
        _INITIALIZING = False
        render()

    def render(*_):
        if _INITIALIZING:
            return
        with output:
            clear_output(wait=True)
            y = target.value
            x = xvar.value
            if y is None:
                print("Selecciona una variable objetivo.")
                return
            if view.value != "Distribución" and x is None:
                print("Selecciona una variable explicativa.")
                return
            df = filter_data(current_df(), year.value, entidad.value, municipio.value, tam_loc.value, est_socio.value, min_obs.value)
            if df.empty:
                print("El filtro no devuelve observaciones.")
                return
            print(f"Observaciones filtradas: {len(df):,}")
            if view.value == "Distribución":
                income_distribution(df, y)
            elif view.value == "X vs ingreso":
                x_vs_income(df, x, y, max_categories.value)
            elif view.value == "Evolución temporal":
                seg = segment.value if segment.value is not None else x
                time_evolution(df, y, seg)
            elif view.value == "Geografía":
                geo = x if x in ["entidad", "municipio", "tam_loc_desc", "est_socio_desc"] else "entidad"
                geography_view(df, y, geo, min_obs.value)

    level.observe(refresh_options, names="value")
    for w in [level, target, year, entidad, municipio, tam_loc, est_socio, min_obs, xvar, segment, view, max_categories]:
        w.observe(render, names="value")
    controls = widgets.VBox([
        widgets.HBox([level, view, year]),
        widgets.HBox([target, xvar, segment]),
        widgets.HBox([entidad, municipio]),
        widgets.HBox([tam_loc, est_socio, min_obs, max_categories]),
    ])
    display(controls, output)
    refresh_options()
else:
    print("Dashboard no inicializado porque faltan dependencias del kernel.")

## 5. Limitaciones

- El modo ponderado se deja pendiente: los marts conservan `factor_hogar`, `est_dis` y `upm`, pero este dashboard no implementa inferencia de diseño muestral.
- Las asociaciones son descriptivas y no implican causalidad.
- Municipio es un filtro exploratorio; siempre se reporta `n` y se permite fijar un mínimo de observaciones.
- Los ingresos derivados de `ingresos.csv` están a nivel persona-clave; para hogar se priorizan variables oficiales de `concentradohogar`.
